# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library.

### Dataset Source

The dataset is described using a Croissant schema, available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's inspect the available record sets (tables), their corresponding `@id`s, and the field `@id`s available within each record set.

If the dataset contains multiple record sets, you'll see all of them listed below, including their fields. All references are strictly by their `@id` for reproducibility.


In [ ]:
# Get all available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.to_json().get('recordSet', [])]
print("Available recordSet @ids:")
for rs_id in record_set_ids:
    print(f"  - {rs_id}")
    # Show fields in each record set
    fields = dataset.record_set(rs_id).field_ids
    print(f"    Fields (@id): {fields}")

if not record_set_ids:
    print("No record sets found in this dataset. If you expect data, check the Croissant schema or data package definition.")

## 3. Data Extraction

We now extract records from a selected record set (table) using the record set and field `@id`s found above. 

All entity references use their `@id` per the FAIR recommendation.


In [ ]:
# List of recordSet @ids from previous code

dataframes = dict()

if not record_set_ids:
    print("No record sets to load. Skipping data extraction.")
else:
    for record_set_id in record_set_ids:
        try:
            # Records are loaded using the record_set's @id
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set {record_set_id}.")
        except Exception as e:
            print(f"Could not load records for {record_set_id}: {e}")

    # Display columns for the first available record set
    if dataframes:
        first_rs = record_set_ids[0]
        print(f"\nColumns for first record set ({first_rs}):")
        print(list(dataframes[first_rs].columns))
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Now we'll demonstrate basic exploratory analysis. We'll select a numeric field (by `@id`) from a record set loaded above, filter records above a threshold, normalize values, and optionally group by another field.

Adjust the `numeric_field_id` and `group_field_id` below according to output from previous code cells, using the appropriate `@id` values as provided in the dataset metadata.


In [ ]:
# ----- Customize this section based on actual recordSet/field ids found previously -----
# Select the record set and a numeric field for demonstration
if dataframes:
    selected_record_set_id = record_set_ids[0]  # Use the first record set, or change as needed
    df = dataframes[selected_record_set_id]
    print(f"Active record set: {selected_record_set_id}")
    print(f"Fields (@id): {list(df.columns)}")

    # Try to heuristically select a numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'iu' or pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].dropna().mean()  # Threshold: mean for demo
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalizing the field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping by a non-numeric field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in this record set for analysis.")
else:
    print("No dataframes available for analysis.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its groupings if available. Update the code to use the correct variable names as discovered in the EDA section.


In [ ]:
# Visualization example for the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in {selected_record_set_id}")
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        # Show grouped mean barplot if grouping was performed
        try:
            plt.figure(figsize=(10,5))
            sns.barplot(
                data=grouped_df.sort_values(numeric_field_id, ascending=False),
                x=group_field_id, y=numeric_field_id, palette="viridis"
            )
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Visualization error: {e}")

## 6. Conclusion

This notebook has shown how to use the `mlcroissant` library to inspect and analyze a dataset described with a Croissant schema, referencing all data component by their `@id`.

- We loaded structured metadata for the FAIR² dataset on rangeland management predictors.
- All record sets, fields, and columns were referenced by their `@id` as best practice.
- We've demonstrated basic data extraction, EDA (filtering, normalization, grouping), and visualized selected fields.

For a real research workflow, after identifying the correct record set and fields, further feature engineering, more advanced statistical or ML analyses, or join/merge with other datasets could be performed—`mlcroissant` provides a standardized way to do so with full provenance tracking and reproducibility.